# Desafio Técnico em Python

## Instalando dependências

In [1]:
! pip install basedosdados -q
import basedosdados as bd
import pandas as pd

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.5/97.5 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 20.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 kB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 41.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━

## Requisitando dataframes

In [2]:
query_chamados_01_04 = "SELECT * FROM `datario.administracao_servicos_publicos.chamado_1746` WHERE DATE(data_inicio) = '2023-04-01'"
query_bairros = "SELECT * FROM `datario.dados_mestres.bairro`"
query_chamados_sossego_2022_2023 = "SELECT * FROM `datario.administracao_servicos_publicos.chamado_1746` WHERE subtipo = 'Perturbação do sossego' AND DATE(data_inicio) BETWEEN '2022-01-01' AND '2023-12-31'"
query_eventos = "SELECT * FROM `datario.turismo_fluxo_visitantes.rede_hoteleira_ocupacao_eventos`"

In [3]:
# Executa queries e cria DataFrames pandas de todos os resultados

# Chamados realizados no dia 01/04/2023
bd_df_chamados_01_04 = bd.read_sql(query_chamados_01_04, billing_project_id="teste-prefeitura-413816")
df_chamados_01_04 = pd.DataFrame(data = bd_df_chamados_01_04)

# Tabela de bairros
bd_df_bairros = bd.read_sql(query_bairros, billing_project_id="teste-prefeitura-413816")
df_bairros = pd.DataFrame(data = bd_df_bairros)

# Chamados entre 2022 e 2023 com subtipo Perturbação do sossego
# Como buscar todos os chamados de 2022 até 2023 para filtrar usando Python e Pandas seria consideravelmente custoso, os filtros necessários foram realizados diretamente na query SQL para este caso
bd_df_chamados_sossego_2022_2023 = bd.read_sql(query_chamados_sossego_2022_2023, billing_project_id="teste-prefeitura-413816")
df_chamados_sossego_2022_2023 = pd.DataFrame(data = bd_df_chamados_sossego_2022_2023)

# Tabela de eventos
bd_df_eventos = bd.read_sql(query_eventos, billing_project_id="teste-prefeitura-413816")
df_eventos = pd.DataFrame(data = bd_df_eventos)

Downloading: 100%|██████████| 4/4 [00:00<00:00, 16.19rows/s]


## Pergunta 1
Quantos chamados foram abertos no dia 01/04/2023?

In [4]:
print("Foram abertos %s chamados no dia 01/04/2023" %(len(df_chamados_01_04.values)))

Foram abertos 73 chamados no dia 01/04/2023


## Pergunta 2
Qual o tipo de chamado que teve mais teve chamados abertos no dia 01/04/2023?


In [5]:
# Agrupa por tipo
group_by_tipo = df_chamados_01_04.groupby('tipo')

# Cria uma série com quantidade de chamados por tipo
count_tipo = group_by_tipo['tipo'].count()

# Nome do tipo com mais chamados na variavel tipo para exibição
tipo = count_tipo[count_tipo == count_tipo.max()].index[0]

# Maior valor da série de quantidade de chamados por tipo
max_value = count_tipo.max()

print("O tipo de chamado mais aberto no dia 01/04/2023 foi o '%s' com %d chamados abertos" % (tipo, count_tipo.max()))

O tipo de chamado mais aberto no dia 01/04/2023 foi o 'Poluição sonora' com 24 chamados abertos


## Pergunta 3
Quais os nomes dos 3 bairros que mais tiveram chamados abertos nesse dia?


In [6]:
# Merge, similar a inner join do SQL, dos DataFrames de Chamados e Bairros
df_merge_chamados_bairros = pd.merge(df_chamados_01_04, df_bairros, on='id_bairro')

# Agrupa os dados por bairro e quantidade de chamados de cada bairro
df_count_chamados_bairros = df_merge_chamados_bairros.groupby('nome').count()

# Seleciona a série de valores de bairro e quantidade de chamados abertos em ordem decrescente
chamados_bairros_decrescente = df_count_chamados_bairros['id_chamado'].sort_values(ascending=False)

print(chamados_bairros_decrescente.iloc[0:10], "\n")

# Armazena os 3 bairros com mais chamados abertos no dia 01/04/2023
resposta_3 = chamados_bairros_decrescente.iloc[0:3]

print("Os 3 bairros com mais chamados abertos neste dia foram:\n")
print(resposta_3)

nome
Engenho de Dentro    8
Campo Grande         6
Leblon               6
Barra da Tijuca      5
Engenho da Rainha    5
Tijuca               3
Santa Teresa         3
Bangu                2
Ipanema              2
Vila da Penha        2
Name: id_chamado, dtype: int64 

Os 3 bairros com mais chamados abertos neste dia foram:

nome
Engenho de Dentro    8
Campo Grande         6
Leblon               6
Name: id_chamado, dtype: int64


## Pergunta 4
Qual o nome da subprefeitura com mais chamados abertos nesse dia?


In [7]:
# Agrupa os dados por bairro e quantidade de chamados de cada bairro
df_count_chamados_subprefeitura = df_merge_chamados_bairros.groupby('subprefeitura').count()

# Seleciona a série de valores de bairro e quantidade de chamados abertos em ordem decrescente
chamados_subprefeitura_decrescente = df_count_chamados_subprefeitura['id_chamado'].sort_values(ascending=False)

# Armazena a subprefeitura com mais chamados abertos no dia 01/04/2023
resposta_4 = chamados_subprefeitura_decrescente.iloc[0:1]


print("A subprefeitura com mais chamados abertos neste dia foi:\n")
print(resposta_4)

A subprefeitura com mais chamados abertos neste dia foi:

subprefeitura
Zona Norte    25
Name: id_chamado, dtype: int64


## Pergunta 5
Existe algum chamado aberto nesse dia que não foi associado a um bairro ou subprefeitura na tabela de bairros? Se sim, por que isso acontece?


In [8]:
df_merge_left_chamados_bairros = pd.merge(df_chamados_01_04, df_bairros, how='left',  on='id_bairro')

In [9]:
# Seleciona chamados com bairro = null, ou seja, chamados não associados a um bairro
df_merge_chamados_bairros_null = df_merge_left_chamados_bairros[df_merge_left_chamados_bairros['nome'].isnull()]
print("Existe um chamado não associado a um bairro, pois seu campo id_bairro é null.\n")
print("Bairro: ",df_merge_chamados_bairros_null['nome'][0], "\n")
print("id_bairro: ",df_merge_chamados_bairros_null['id_bairro'][0], "\n")
df_merge_chamados_bairros_null[['id_chamado', 'id_bairro', 'nome', 'subprefeitura']]

Existe um chamado não associado a um bairro, pois seu campo id_bairro é null.

Bairro:  nan 

id_bairro:  None 



,id_chamado,id_bairro,nome,subprefeitura
0,18516246,None,NaN,NaN


In [10]:
# Seleciona chamados com subprefeitura = null, ou seja, chamados não associados a subprefeitura
df_merge_chamados_bairros_subprefeitura_null = df_merge_left_chamados_bairros[df_merge_left_chamados_bairros['subprefeitura'].isnull()]

print("O mesmo chamado também não foi associado a uma subprefeitura, pelo mesmo motivo, e é o único não associado a uma subprefeitura.\n")
df_merge_chamados_bairros_subprefeitura_null

O mesmo chamado também não foi associado a uma subprefeitura, pelo mesmo motivo, e é o único não associado a uma subprefeitura.



,id_chamado,data_inicio,data_fim,id_bairro,id_territorialidade,id_logradouro,numero_logradouro,id_unidade_organizacional,nome_unidade_organizacional,id_unidade_organizacional_mae,...,id_area_planejamento,id_regiao_planejamento,nome_regiao_planejamento,id_regiao_administrativa,nome_regiao_administrativa,subprefeitura,area,perimetro,geometry_wkt,geometry_y
0,18516246,2023-04-01 00:55:38,2023-04-01 00:55:38,None,None,None,<NA>,1706,TR/SUBOP/CFT - Coordenadoria de Fiscalização e...,SMTR - Secretaria Municipal de Transportes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Pergunta 6
Quantos chamados com o subtipo "Perturbação do sossego" foram abertos desde 01/01/2022 até 31/12/2023 (incluindo extremidades)?

In [11]:
print("Como buscar todos os chamados de 2022 até 2023 seria consideravelmente custoso, os filtros necessários foram realizados diretamente na query SQL para este caso\n")
print("O número de chamados com subtipo 'Perturbação do sossego' abertos desde 01/01/2022 até 31/12/2023 é:", len(df_chamados_sossego_2022_2023))

Como buscar todos os chamados de 2022 até 2023 seria consideravelmente custoso, os filtros necessários foram realizados diretamente na query SQL para este caso

O número de chamados com subtipo 'Perturbação do sossego' abertos desde 01/01/2022 até 31/12/2023 é: 42408


## Pergunta 7
Selecione os chamados com esse subtipo que foram abertos durante os eventos contidos na tabela de eventos (Reveillon, Carnaval e Rock in Rio).


In [41]:
# A dificuldade de retornar este JOIN de Chamados com o Evento de acordo com a data_inicio me fez optar por realizar a query diretamente do SQL
query_chamados_sossego_eventos = """
SELECT *
FROM `datario.administracao_servicos_publicos.chamado_1746` Chamados
INNER JOIN `datario.turismo_fluxo_visitantes.rede_hoteleira_ocupacao_eventos` Eventos
ON (DATE(Chamados.data_inicio) BETWEEN Eventos.data_inicial AND Eventos.data_final)
WHERE subtipo = "Perturbação do sossego"
"""

bd_df_chamados_sossego_eventos = bd.read_sql(query_chamados_sossego_eventos, billing_project_id="teste-prefeitura-413816")
df_chamados_sossego_eventos = pd.DataFrame(data = bd_df_chamados_sossego_eventos)

Downloading: 100%|██████████| 1212/1212 [00:00<00:00, 1785.19rows/s]


In [42]:
print("Os chamados de subtipo 'Perturbação do sossego' abertos durante os eventos contidos na tabela de eventos foram:\n")
df_chamados_sossego_eventos[['data_inicio', 'data_inicial', 'data_final', 'evento']]

Os chamados de subtipo 'Perturbação do sossego' abertos durante os eventos contidos na tabela de eventos foram:



,data_inicio,data_inicial,data_final,evento
0,2023-02-18 17:07:39,2023-02-18,2023-02-21,Carnaval
1,2023-02-21 12:53:07,2023-02-18,2023-02-21,Carnaval
2,2023-02-20 16:44:59,2023-02-18,2023-02-21,Carnaval
3,2023-02-19 02:39:03,2023-02-18,2023-02-21,Carnaval
4,2023-02-18 08:24:30,2023-02-18,2023-02-21,Carnaval
...,...,...,...,...
1207,2022-09-11 09:07:06,2022-09-08,2022-09-11,Rock in Rio
1208,2022-09-11 04:23:30,2022-09-08,2022-09-11,Rock in Rio
1209,2022-09-02 12:29:46,2022-09-02,2022-09-04,Rock in Rio
1210,2022-09-11 18:05:34,2022-09-08,2022-09-11,Rock in Rio


## Pergunta 8
Quantos chamados desse subtipo foram abertos em cada evento?


In [43]:
resposta_8 = df_chamados_sossego_eventos.groupby(['evento']).count()

print("O número de chamados desse subtipo abertos em cada evento foram: \n")
resposta_8['id_chamado']

O número de chamados desse subtipo abertos em cada evento foram: 



evento
Carnaval       241
Reveillon      137
Rock in Rio    834
Name: id_chamado, dtype: int64

## Pergunta 9
Qual evento teve a maior média diária de chamados abertos desse subtipo?

In [44]:
df_chamados_sossego_date = df_chamados_sossego_eventos.copy()
df_chamados_sossego_date['data_inicio'] = df_chamados_sossego_date['data_inicio'].dt.date

In [68]:
df_chamados_por_dia = df_chamados_sossego_date.groupby(['data_inicio', 'evento']).count()
resposta_9 = df_chamados_por_dia.groupby('evento').mean()

series_medias = resposta_9['id_chamado']
nome_evento = series_medias[series_medias == series_medias.max()].index[0]
print("O evento de maior média diária de chamados abertos de subtipo 'Perturbação do sossego' foi: %s com %d chamados abertos por dia\n" %(nome_evento, series_medias.max()))
series_medias
# resposta_9 = df_chamados_por_dia.groupby('evento').mean()
# resposta_9

O evento de maior média diária de chamados abertos de subtipo 'Perturbação do sossego' foi: Rock in Rio com 119 chamados abertos por dia



evento
Carnaval        60.250000
Reveillon       45.666667
Rock in Rio    119.142857
Name: id_chamado, dtype: float64

## Pergunta 10
Compare as médias diárias de chamados abertos desse subtipo durante os eventos específicos (Reveillon, Carnaval e Rock in Rio) e a média diária de chamados abertos desse subtipo considerando todo o período de 01/01/2022 até 31/12/2023.

In [49]:
df_chamados_sossego_2022_2023_date = df_chamados_sossego_2022_2023.copy()
df_chamados_sossego_2022_2023_date['data_inicio'] = df_chamados_sossego_2022_2023_date['data_inicio'].dt.date

In [69]:
# Média diária de chamados abertos entre 2022 e 2023
df_media_diaria_22_23 = df_chamados_sossego_2022_2023_date.groupby(['data_inicio']).count().mean()

print("A média diária de chamados abertos entre 2022 e 2023 foi de %.2f, próxima à média diária de chamados de subtipo 'Perturbação do sossego' abertos no Carnaval (%.2f), acima da média do Reveillon (%.2f) e abaixo da média do Rock in Rio (%.2f)" %(df_media_diaria_22_23.max(), series_medias['Carnaval'], series_medias['Reveillon'], series_medias['Rock in Rio']))

A média diária de chamados abertos entre 2022 e 2023 foi de 63.20, próxima à média diária de chamados de subtipo 'Perturbação do sossego' abertos no Carnaval (60.25), acima da média do Reveillon (45.67) e abaixo da média do Rock in Rio (119.14)
